In [2]:
!pip install tensorflow opencv-python numpy pillow scikit-learn openpyxl

In [3]:
import os, random, json, csv
import numpy as np
import cv2
from PIL import Image
from datetime import datetime
import openpyxl

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics.pairwise import cosine_similarity

In [4]:
STUDENTS_FOLDER = 'Faces'
IMG_SIZE        = (128, 128)
SEED            = 42

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

In [5]:
def load_img(path):
    img = Image.open(path).convert('RGB').resize(IMG_SIZE)
    return np.array(img).astype(np.float32) / 255.0

X      = []
labels = []

for filename in os.listdir(STUDENTS_FOLDER):
    if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
        path = os.path.join(STUDENTS_FOLDER, filename)
        name = os.path.splitext(filename)[0]
        name = '_'.join(name.split('_')[:-1])
        X.append(load_img(path))
        labels.append(name)

X = np.array(X)
print('Dataset shape:', X.shape)
print('Unique students:', set(labels))

Dataset shape: (2377, 128, 128, 3)
Unique students: {'Camila Cabello', 'Vijay Deverakonda', 'Hugh Jackman', 'Courtney Cox', 'Henry Cavill', 'Andy Samberg', 'Jessica Alba', 'Charlize Theron', 'Hrithik Roshan', 'Priyanka Chopra', 'Lisa Kudrow', 'Ellen Degeneres', 'Anushka Sharma', 'Dwayne Johnson', 'Robert Downey Jr', 'Amitabh Bachchan', 'Billie Eilish', 'Elizabeth Olsen', 'Marmik', 'Natalie Portman', 'Claire Holt', 'Virat Kohli', 'Margot Robbie', 'Tom Cruise', 'Roger Federer', 'Kashyap', 'Zac Efron', 'Brad Pitt', 'Alia Bhatt'}


In [6]:
le      = LabelEncoder()
y       = le.fit_transform(labels)
n_class = len(le.classes_)

print('Number of students:', n_class)
print('Classes:', le.classes_)

Number of students: 29
Classes: ['Alia Bhatt' 'Amitabh Bachchan' 'Andy Samberg' 'Anushka Sharma'
 'Billie Eilish' 'Brad Pitt' 'Camila Cabello' 'Charlize Theron'
 'Claire Holt' 'Courtney Cox' 'Dwayne Johnson' 'Elizabeth Olsen'
 'Ellen Degeneres' 'Henry Cavill' 'Hrithik Roshan' 'Hugh Jackman'
 'Jessica Alba' 'Kashyap' 'Lisa Kudrow' 'Margot Robbie' 'Marmik'
 'Natalie Portman' 'Priyanka Chopra' 'Robert Downey Jr' 'Roger Federer'
 'Tom Cruise' 'Vijay Deverakonda' 'Virat Kohli' 'Zac Efron']


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.15, random_state=SEED
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.15, random_state=SEED
)

print('Train:', len(X_train))
print('Val  :', len(X_val))
print('Test :', len(X_test))

Train: 1717
Val  : 303
Test : 357


In [8]:
inp = Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
print('Input shape    :', inp.shape)

x = Conv2D(32, 3, activation='relu', padding='same')(inp)
print('After Conv2D(32)  :', x.shape)

x = MaxPooling2D()(x)
print('After MaxPooling  :', x.shape)

x = Conv2D(64, 3, activation='relu', padding='same')(x)
print('After Conv2D(64)  :', x.shape)

x = MaxPooling2D()(x)
print('After MaxPooling  :', x.shape)

x = Conv2D(128, 3, activation='relu', padding='same')(x)
print('After Conv2D(128) :', x.shape)

x = MaxPooling2D()(x)
print('After MaxPooling  :', x.shape)

x = Flatten()(x)
print('After Flatten     :', x.shape)

x = Dropout(0.3)(x)

x   = Dense(128, activation='relu')(x)
print('After Dense(128)  :', x.shape)

out = Dense(n_class, activation='softmax')(x)
print('Output shape      :', out.shape)

Input shape    : (None, 128, 128, 3)
After Conv2D(32)  : (None, 128, 128, 32)
After MaxPooling  : (None, 64, 64, 32)
After Conv2D(64)  : (None, 64, 64, 64)
After MaxPooling  : (None, 32, 32, 64)
After Conv2D(128) : (None, 32, 32, 128)
After MaxPooling  : (None, 16, 16, 128)
After Flatten     : (None, 32768)
After Dense(128)  : (None, 128)
Output shape      : (None, 29)


In [9]:
model = Model(inp, out)
model.compile(
    optimizer = 'adam',
    loss      = 'sparse_categorical_crossentropy',
    metrics   = ['accuracy']
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 128, 128, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 64, 64, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 32, 32, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 32768)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32768)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     4,194,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 29)             │         3,741 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,291,421 (16.37 MB)

 Trainable params: 4,291,421 (16.37 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = model.fit(
    X_train, y_train,
    epochs          = 20,
    batch_size      = 8,
    validation_data = (X_val, y_val)
)

Epoch 1/20
215/215 ━━━━━━━━━━━━━━━━━━━━ 22s 88ms/step - accuracy: 0.0751 - loss: 3.2826 - val_accuracy: 0.0990 - val_loss: 3.1328
Epoch 2/20
215/215 ━━━━━━━━━━━━━━━━━━━━ 19s 88ms/step - accuracy: 0.3023 - loss: 2.4082 - val_accuracy: 0.3729 - val_loss: 2.2359
Epoch 3/20
215/215 ━━━━━━━━━━━━━━━━━━━━ 17s 78ms/step - accuracy: 0.5457 - loss: 1.5215 - val_accuracy: 0.5380 - val_loss: 1.6734
Epoch 4/20
215/215 ━━━━━━━━━━━━━━━━━━━━ 14s 64ms/step - accuracy: 0.7169 - loss: 0.9318 - val_accuracy: 0.5842 - val_loss: 1.6253
Epoch 5/20
215/215 ━━━━━━━━━━━━━━━━━━━━ 13s 61ms/step - accuracy: 0.7985 - loss: 0.6062 - val_accuracy: 0.5743 - val_loss: 1.7966
Epoch 6/20
215/215 ━━━━━━━━━━━━━━━━━━━━ 13s 61ms/step - accuracy: 0.8818 - loss: 0.3466 - val_accuracy: 0.5908 - val_loss: 2.0897
Epoch 7/20
170/215 ━━━━━━━━━━━━━━━━━━━━ 2s 58ms/step - accuracy: 0.9061 - loss: 0.2816

In [ ]:
loss, acc = model.evaluate(X_test, y_test, verbose=0)
print('Test accuracy:', acc)

y_pred = np.argmax(model.predict(X_test), axis=1)
print('\nReport:\n', classification_report(y_test, y_pred, target_names=le.classes_))

Test accuracy: 0.6190476417541504
12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 53ms/step

Report:
                    precision    recall  f1-score   support

       Alia Bhatt       0.33      0.50      0.40         2
 Amitabh Bachchan       1.00      0.71      0.83        14
     Andy Samberg       0.83      0.50      0.62        10
   Anushka Sharma       0.67      0.40      0.50        15
    Billie Eilish       0.62      0.57      0.59        14
        Brad Pitt       0.72      0.59      0.65        22
   Camila Cabello       0.69      0.75      0.72        12
  Charlize Theron       0.67      0.40      0.50        20
      Claire Holt       0.38      0.38      0.38         8
     Courtney Cox       0.42      0.71      0.53         7
   Dwayne Johnson       0.80      0.40      0.53        10
  Elizabeth Olsen       0.50      0.14      0.22         7
  Ellen Degeneres       0.50      0.30      0.38        10
     Henry Cavill       0.67      0.59      0.62        17
   Hrithik Roshan       0.55  

In [ ]:
# نبني feature extractor من الـ model
# نشيل الـ output layer ونستخدم الـ Dense(128) كـ output
# يعني بدل ما يقول اسم، يحول الوجه لـ 128 رقم تمثّل شكله
feature_extractor = Model(inputs=model.input, outputs=model.layers[-2].output)
print('Feature extractor ready!')

Feature extractor ready!


In [ ]:
# نحسب الـ features لكل صورة في الـ Students folder
# هذي تصير قاعدة البيانات اللي نقارن فيها وجوه الكاميرا
student_features = {}

for filename in os.listdir('Students_Images'):
    if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
        path = os.path.join('Students_Images', filename)
        name = os.path.splitext(filename)[0]
        name = '_'.join(name.split('_')[:-1])

        img = load_img(path)
        img = np.expand_dims(img, axis=0)

        # نستخرج الـ features من الصورة
        features = feature_extractor.predict(img, verbose=0).flatten()

        # إذا الطالب عنده أكثر من صورة نحسب المتوسط
        if name in student_features:
            student_features[name] = (student_features[name] + features) / 2
        else:
            student_features[name] = features

class_names = list(student_features.keys())
print('Students:', class_names)

Students: ['Akshay Kumar', '']


In [ ]:
# identify_face: تستخدم الـ feature extractor
# تحول وجه الكاميرا لأرقام وتقارنها مع أرقام كل طالب
# هذا نفس مفهوم cosine_similarity اللي شرحه الأستاذ في image-search
def identify_face(frame):
    img = Image.fromarray(frame).convert('RGB').resize(IMG_SIZE)
    img = np.array(img).astype(np.float32) / 255.0
    img = np.expand_dims(img, axis=0)

    # نستخرج الـ features من الـ frame
    frame_features = feature_extractor.predict(img, verbose=0).flatten()

    # نقارن مع كل طالب ونلاقي الأكثر تشابهاً
    best_name  = None
    best_score = 0

    for name, features in student_features.items():
        score = cosine_similarity(
            frame_features.reshape(1, -1),
            features.reshape(1, -1)
        )[0][0]

        if score > best_score:
            best_score = score
            best_name  = name

    # إذا التشابه أقل من 80% يعني الـ model مو متأكد
    if best_score < 0.8:
        return None

    return best_name

In [ ]:
def mark_attendance(name, status, csv_file='attendance.csv'):
    now      = datetime.now()
    date_str = now.strftime('%Y-%m-%d')
    time_str = now.strftime('%H:%M:%S') if status != 'Absent' else '---'

    if os.path.exists(csv_file):
        with open(csv_file, 'r', encoding='utf-8-sig') as f:
            reader = csv.DictReader(f)
            for row in reader:
                if row.get('Name') == name and row.get('Date') == date_str:
                    return False

    file_exists = os.path.exists(csv_file)
    with open(csv_file, 'a', newline='', encoding='utf-8-sig') as f:
        fieldnames = ['Name', 'Date', 'Arrival_Time', 'Status']
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if not file_exists:
            writer.writeheader()
        writer.writerow({
            'Name'        : name,
            'Date'        : date_str,
            'Arrival_Time': time_str,
            'Status'      : status
        })
    return True

In [ ]:
LATE_HOUR   = 8
LATE_MINUTE = 15

cap    = cv2.VideoCapture(0)
count  = 0
logged = {}

while True:
    success, frame = cap.read()
    if not success:
        break

    count += 1

    if count % 30 == 0:
        name = identify_face(frame)

        if name and name not in logged:
            now = datetime.now()
            if now.hour < LATE_HOUR or (now.hour == LATE_HOUR and now.minute < LATE_MINUTE):
                status = 'Present'
            else:
                status = 'Late'

            mark_attendance(name, status)
            logged[name] = status
            print(name)
            print(status)

    now_str = datetime.now().strftime('%H:%M:%S')
    cv2.putText(frame, f'Time: {now_str}', (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
    cv2.putText(frame, f'Present: {len(logged)}/{len(class_names)}', (10, 60),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

    y = 100
    for n, s in logged.items():
        color = (0, 255, 0) if s == 'Present' else (0, 165, 255)
        cv2.putText(frame, f'{n}: {s}', (10, y),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
        y += 25

    cv2.imshow('Attendance System', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

In [ ]:
for name in class_names:
    if name in logged:
        print(name, '|', logged[name])
    else:
        print(name, '| Absent')
        mark_attendance(name, 'Absent')

print('='*40)
print('Present :', sum(1 for v in logged.values() if v == 'Present'))
print('Late    :', sum(1 for v in logged.values() if v == 'Late'))
print('Absent  :', len([n for n in class_names if n not in logged]))
print('='*40)

Akshay Kumar | Absent
 | Absent
Present : 0
Late    : 0
Absent  : 2


In [ ]:
def save_to_excel(class_names, logged, filename='attendance.xlsx'):
    wb = openpyxl.Workbook()
    ws = wb.active
    ws.title = 'Attendance'
    ws.append(['Name', 'Date', 'Arrival Time', 'Status'])

    date_str = datetime.now().strftime('%Y-%m-%d')

    for name in class_names:
        status  = logged.get(name, 'Absent')
        arrival = '---'

        if os.path.exists('attendance.csv'):
            with open('attendance.csv', 'r', encoding='utf-8-sig') as f:
                reader = csv.DictReader(f)
                for row in reader:
                    if row.get('Name') == name and row.get('Date') == date_str:
                        arrival = row.get('Arrival_Time', '---')
                        break

        ws.append([name, date_str, arrival, status])

    wb.save(filename)
    print('saved to', filename)

save_to_excel(class_names, logged)

saved to attendance.xlsx


In [ ]:
import json

model.save('face_model.keras')

embeddings = {name: feat.tolist() for name, feat in student_features.items()}
with open('student_features.json', 'w') as f:
    json.dump(embeddings, f)

print('saved!')

NameError: name 'model' is not defined